# 01 - Data Loading & Feature Engineering

Pipeline de carga, limpeza e criação de features para os 3.9M de registros.

**Output:** `processed/crash_features.parquet` - dataset pronto para análise.

In [11]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
from datetime import datetime

print(f"DB: {DB_PATH}")
print(f"Existe: {DB_PATH.exists()}")

DB: c:\IA\Crash_AI\data\db\brabet_crash_COMPLETO.db
Existe: True


## 1. Carga dos Dados Brutos

In [12]:
df = load_raw_data()
print(f"Shape: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nTipos:\n{df.dtypes}")
df.head(10)

Carregados 3,964,265 registros via DuckDB
Shape: (3964265, 7)

Colunas: ['id', 'date', 'time', 'multiplicador', 'result', 'tipo', 'externalId']

Tipos:
id                object
date              object
time              object
multiplicador    float64
result            object
tipo              object
externalId        object
dtype: object


,id,date,time,multiplicador,result,tipo,externalId
0,0184f17b-c3f8-75aa-8653-e12c87004dc7,2022-12-08T08:25:31.000-0300,08:25:31,1.09,"1,09x",LOW,19857
1,0184f17b-fe90-7212-9ae9-5b18b1130e41,2022-12-08T08:25:46.000-0300,08:25:46,1.57,"1,57x",LOW,19858
2,0184f17c-5098-7541-bc96-83c74c5af93d,2022-12-08T08:26:07.000-0300,08:26:07,1.41,"1,41x",LOW,19859
3,0184f17c-9ad0-7d7d-9f52-6f7f065cde38,2022-12-08T08:26:26.000-0300,08:26:26,1.09,"1,09x",LOW,19860
4,0184f17c-d950-7f21-a30a-19b06666c45f,2022-12-08T08:26:42.000-0300,08:26:42,2.75,"2,75x",HIGH,19861
5,0184f17d-3af8-7cd1-ac78-0d4bbf0c1cef,2022-12-08T08:27:07.000-0300,08:27:07,9.99,"9,99x",HIGH,19862
6,0184f17d-d350-7989-abdc-b16d16d98740,2022-12-08T08:27:46.000-0300,08:27:46,1.17,"1,17x",LOW,19863
7,0184f17e-15b8-7dbb-b233-78ad72ae42f4,2022-12-08T08:28:03.000-0300,08:28:03,4.20,"4,20x",HIGH,19864
8,0184f17e-8318-781a-86c5-3c944b37fb49,2022-12-08T08:28:31.000-0300,08:28:31,2.37,"2,37x",HIGH,19865
9,0184f17e-e0d8-71db-9032-82f6da624ab4,2022-12-08T08:28:55.000-0300,08:28:55,8.45,"8,45x",HIGH,19866


## 2. Limpeza e Tipagem

In [13]:
# Converter date para datetime
df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')

# Remover linhas sem data ou multiplicador
before = len(df)
df = df.dropna(subset=['date', 'multiplicador'])
after = len(df)
print(f"Removidas {before - after} linhas com NaN ({(before-after)/before*100:.2f}%)")

# Garantir ordem cronológica
df = df.sort_values('date').reset_index(drop=True)

# Extrair componentes temporais
dt = df['date'].dt  # type: ignore[union-attr]
df['hora'] = dt.hour
df['dia_semana'] = dt.dayofweek  # 0=Monday, 6=Sunday
df['dia_semana_nome'] = dt.day_name()
df['mes'] = dt.month
df['ano'] = dt.year
df['dia_do_ano'] = dt.dayofyear

print(f"\nPeríodo: {df['date'].min()} a {df['date'].max()}")
print(f"Total: {len(df):,} registros")
df.head()

Removidas 0 linhas com NaN (0.00%)

Período: 2022-12-08 11:25:31+00:00 a 2026-02-04 11:23:05+00:00
Total: 3,964,265 registros


,id,date,time,multiplicador,result,tipo,externalId,hora,dia_semana,dia_semana_nome,mes,ano,dia_do_ano
0,0184f17b-c3f8-75aa-8653-e12c87004dc7,2022-12-08 11:25:31+00:00,08:25:31,1.09,"1,09x",LOW,19857,11,3,Thursday,12,2022,342
1,0184f17b-fe90-7212-9ae9-5b18b1130e41,2022-12-08 11:25:46+00:00,08:25:46,1.57,"1,57x",LOW,19858,11,3,Thursday,12,2022,342
2,0184f17c-5098-7541-bc96-83c74c5af93d,2022-12-08 11:26:07+00:00,08:26:07,1.41,"1,41x",LOW,19859,11,3,Thursday,12,2022,342
3,0184f17c-9ad0-7d7d-9f52-6f7f065cde38,2022-12-08 11:26:26+00:00,08:26:26,1.09,"1,09x",LOW,19860,11,3,Thursday,12,2022,342
4,0184f17c-d950-7f21-a30a-19b06666c45f,2022-12-08 11:26:42+00:00,08:26:42,2.75,"2,75x",HIGH,19861,11,3,Thursday,12,2022,342


## 3. Feature Engineering - Classificações

In [14]:
# Classificação LOW/HIGH
df['is_low'] = (df['multiplicador'] < LOW_THRESHOLD).astype(int)
df['is_high'] = 1 - df['is_low']

print(f"LOW  (< {LOW_THRESHOLD}x): {df['is_low'].sum():,} ({df['is_low'].mean()*100:.1f}%)")
print(f"HIGH (>= {LOW_THRESHOLD}x): {df['is_high'].sum():,} ({df['is_high'].mean()*100:.1f}%)")

LOW  (< 2.0x): 2,162,543 (54.6%)
HIGH (>= 2.0x): 1,801,722 (45.4%)


## 4. Feature Engineering - Streaks

In [15]:
# Calcular streaks de LOW e HIGH
def compute_streaks(series):
    """Calcula streak corrente para cada posição."""
    streaks = np.zeros(len(series), dtype=int)
    if len(series) == 0:
        return streaks
    streaks[0] = int(series.iloc[0])
    vals = series.values
    for i in range(1, len(vals)):
        if vals[i] == 1:
            streaks[i] = streaks[i-1] + 1
        else:
            streaks[i] = 0
    return streaks

print("Calculando low_streak...")
df['low_streak'] = compute_streaks(df['is_low'])

print("Calculando high_streak...")
df['high_streak'] = compute_streaks(df['is_high'])

print(f"\nMax LOW streak:  {df['low_streak'].max()}")
print(f"Max HIGH streak: {df['high_streak'].max()}")
print(f"\nDistribuição de LOW streaks (top 10):")
print(df[df['low_streak'] > 0]['low_streak'].value_counts().sort_index().tail(15))

Calculando low_streak...
Calculando high_streak...

Max LOW streak:  19
Max HIGH streak: 18

Distribuição de LOW streaks (top 10):
low_streak
5     89297
6     49106
7     22508
8      9727
9      4302
10     1830
11      787
12      292
13      102
14       36
15       15
16        8
17        3
18        1
19        1
Name: count, dtype: int64


## 5. Feature Engineering - Rolling Statistics

In [16]:
mult = df['multiplicador']

for w in ROLLING_WINDOWS:
    print(f"  Rolling window {w}...")
    df[f'rolling_mean_{w}'] = mult.rolling(w, min_periods=1).mean()
    df[f'rolling_std_{w}'] = mult.rolling(w, min_periods=1).std().fillna(0)
    df[f'rolling_median_{w}'] = mult.rolling(w, min_periods=1).median()
    df[f'pct_low_{w}'] = df['is_low'].rolling(w, min_periods=1).mean()

print("Rolling statistics concluído.")

  Rolling window 5...
  Rolling window 10...
  Rolling window 20...
  Rolling window 50...
  Rolling window 100...
  Rolling window 250...
Rolling statistics concluído.


## 6. Feature Engineering - Lags

In [17]:
for lag in LAG_PERIODS:
    df[f'lag_{lag}'] = mult.shift(lag)

# Log-return
df['log_return'] = np.log(mult / mult.shift(1))

# Diferença absoluta
df['diff_1'] = mult.diff(1)

print(f"Features de lag criadas: lag_1 a lag_{LAG_PERIODS[-1]}")
print(f"Total de colunas: {len(df.columns)}")

Features de lag criadas: lag_1 a lag_10
Total de colunas: 53


## 7. Feature Engineering - Marcadores de Tragédia

In [18]:
# Marcar tragédias (12+ LOWs consecutivos)
df['is_tragedy'] = (df['low_streak'] >= TRAGEDY_STREAK).astype(int)

# Marcar pré-tragédia (250 rounds antes de cada tragédia)
tragedy_indices = df[df['is_tragedy'] == 1].index.tolist()
pre_tragedy_mask = pd.Series(False, index=df.index)
for idx in tragedy_indices:
    start = max(0, idx - 250)
    pre_tragedy_mask.iloc[start:idx] = True

df['is_pre_tragedy'] = pre_tragedy_mask.astype(int)

# Clima preditivo
df['clima'] = 'Normal'
df.loc[df['is_pre_tragedy'] == 1, 'clima'] = 'Pre_Tragedy'
df.loc[df['is_tragedy'] == 1, 'clima'] = 'In_Tragedy'

print(f"\nTragédias encontradas: {df['is_tragedy'].sum()}")
print(f"Rounds pré-tragédia: {df['is_pre_tragedy'].sum()}")
print(f"\nDistribuição de climas:")
print(df['clima'].value_counts())


Tragédias encontradas: 458
Rounds pré-tragédia: 72247

Distribuição de climas:
clima
Normal         3891735
Pre_Tragedy      72072
In_Tragedy         458
Name: count, dtype: int64


## 8. Resumo do Dataset Processado

In [19]:
print(f"Shape final: {df.shape}")
print(f"\nColunas ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col} ({df[col].dtype})")

print(f"\n--- Estatísticas do multiplicador ---")
print(df['multiplicador'].describe())

print(f"\n--- Memória ---")
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"RAM: {mem_mb:.1f} MB")

Shape final: (3964265, 56)

Colunas (56):
   1. id (object)
   2. date (datetime64[ns, UTC])
   3. time (object)
   4. multiplicador (float64)
   5. result (object)
   6. tipo (object)
   7. externalId (object)
   8. hora (int32)
   9. dia_semana (int32)
  10. dia_semana_nome (object)
  11. mes (int32)
  12. ano (int32)
  13. dia_do_ano (int32)
  14. is_low (int32)
  15. is_high (int32)
  16. low_streak (int32)
  17. high_streak (int32)
  18. rolling_mean_5 (float64)
  19. rolling_std_5 (float64)
  20. rolling_median_5 (float64)
  21. pct_low_5 (float64)
  22. rolling_mean_10 (float64)
  23. rolling_std_10 (float64)
  24. rolling_median_10 (float64)
  25. pct_low_10 (float64)
  26. rolling_mean_20 (float64)
  27. rolling_std_20 (float64)
  28. rolling_median_20 (float64)
  29. pct_low_20 (float64)
  30. rolling_mean_50 (float64)
  31. rolling_std_50 (float64)
  32. rolling_median_50 (float64)
  33. pct_low_50 (float64)
  34. rolling_mean_100 (float64)
  35. rolling_std_100 (float64)
  

## 9. Salvar em Parquet

In [20]:
# Remover linhas com NaN nos lags (primeiras 10 linhas)
df_clean = df.dropna(subset=[f'lag_{LAG_PERIODS[-1]}'])
print(f"Removidas {len(df) - len(df_clean)} linhas iniciais com NaN nos lags")

# Salvar
df_clean.to_parquet(PARQUET_FEATURES, index=False)
size_mb = PARQUET_FEATURES.stat().st_size / 1024**2
print(f"\nSalvo em: {PARQUET_FEATURES}")
print(f"Tamanho: {size_mb:.1f} MB")
print(f"Registros: {len(df_clean):,}")

Removidas 10 linhas iniciais com NaN nos lags

Salvo em: c:\IA\Crash_AI\data\processed\crash_features.parquet
Tamanho: 736.5 MB
Registros: 3,964,255
